In [1]:
# Cell 1 — Imports
import torch
import numpy as np
import pandas as pd
import plotly.graph_objects as go
import plotly.express as px
from transformer_lens import HookedTransformer
import einops
import warnings
warnings.filterwarnings('ignore')

print("All imports successful")
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available:  {torch.cuda.is_available()}")

c:\Users\Gurveer\anaconda3\envs\ds-portfolio\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


All imports successful
PyTorch version: 2.11.0+cpu
CUDA available:  False


In [9]:
# Cell 2 — Load GPT-2 Medium Model
print("Loading GPT-2 Medium via TransformerLens...")
model = HookedTransformer.from_pretrained("gpt2-medium")
model.eval()

print(f"\nModel loaded successfully")
print(f"Layers:          {model.cfg.n_layers}")
print(f"Attention heads: {model.cfg.n_heads}")
print(f"d_model:         {model.cfg.d_model}")
print(f"d_head:          {model.cfg.d_head}")
print(f"Vocab size:      {model.cfg.d_vocab}")

# Test forward pass
test_prompt = "The Eiffel Tower is located in the city of"
tokens = model.to_tokens(test_prompt)
logits = model(tokens)

# Show top 5 predictions
top5 = logits[0, -1].topk(5)
print(f"\nTest prompt: '{test_prompt}'")
print(f"Top 5 predicted tokens:")
for val, idx in zip(top5.values, top5.indices):
    print(f"  '{model.to_string(idx)}' — logit: {val:.2f}")

Loading GPT-2 Medium via TransformerLens...
Loaded pretrained model gpt2-medium into HookedTransformer

Model loaded successfully
Layers:          24
Attention heads: 16
d_model:         1024
d_head:          64
Vocab size:      50257

Test prompt: 'The Eiffel Tower is located in the city of'
Top 5 predicted tokens:
  ' Paris' — logit: 20.47
  ' Lyon' — logit: 17.70
  ' Nice' — logit: 17.64
  ' E' — logit: 15.58
  ' T' — logit: 15.50


In [10]:
# Cell 3 — Attention Pattern Analysis
# Analyze what each attention head attends to

prompt = "When John and Mary went to the store, John gave the bag to"
tokens = model.to_tokens(prompt)
token_strs = [model.to_string(t) for t in tokens[0]]

print(f"Prompt: '{prompt}'")
print(f"\nTokens: {token_strs}")

# Run with cache to capture all activations
logits, cache = model.run_with_cache(tokens)

predicted = logits[0, -1].argmax()
print(f"\nPredicted next token: '{model.to_string(predicted)}'")

# Extract attention patterns from all layers
print(f"\nExtracting attention patterns...")
attention_patterns = []
for layer in range(model.cfg.n_layers):
    attn = cache['pattern', layer]  # shape: [batch, heads, seq, seq]
    attention_patterns.append(attn[0].detach().numpy())

print(f"Captured {len(attention_patterns)} layers of attention")
print(f"Pattern shape per layer: {attention_patterns[0].shape}")
print(f"(heads, seq_len, seq_len) = {attention_patterns[0].shape}")

Prompt: 'When John and Mary went to the store, John gave the bag to'

Tokens: ['<|endoftext|>', 'When', ' John', ' and', ' Mary', ' went', ' to', ' the', ' store', ',', ' John', ' gave', ' the', ' bag', ' to']

Predicted next token: ' Mary'

Extracting attention patterns...
Captured 24 layers of attention
Pattern shape per layer: (16, 15, 15)
(heads, seq_len, seq_len) = (16, 15, 15)


In [11]:
# Cell 4 — Visualize Attention Heads
# Find which heads attend to names (John/Mary)
seq_len  = len(token_strs)
n_layers = model.cfg.n_layers
n_heads  = model.cfg.n_heads

# Plot attention pattern for layer 0, all heads
layer_to_plot = 0
fig = go.Figure()

for head in range(n_heads):
    attn = attention_patterns[layer_to_plot][head]
    fig.add_trace(go.Heatmap(
        z=attn,
        x=token_strs,
        y=token_strs,
        colorscale='Blues',
        showscale=(head == 0),
        visible=(head == 0),
        name=f'Head {head}'
    ))

# Dropdown to switch between heads
buttons = []
for head in range(n_heads):
    visible = [False] * n_heads
    visible[head] = True
    buttons.append(dict(
        label=f'Head {head}',
        method='update',
        args=[{'visible': visible},
              {'title': f'Layer {layer_to_plot} Head {head} Attention Pattern'}]
    ))

fig.update_layout(
    title=f'Layer {layer_to_plot} — Attention Patterns (select head)',
    updatemenus=[dict(
        active=0,
        buttons=buttons,
        x=1.15, y=1.0
    )],
    template='plotly_dark',
    width=850, height=600
)
fig.show()
print("Use dropdown to explore all 12 attention heads")

Use dropdown to explore all 12 attention heads


In [6]:
# Cell 5 — Activation Patching (Core Interpretability Technique)
# This is the key technique used in Anthropic's research
# We find which attention heads are responsible for the IOI task

clean_prompt    = "When John and Mary went to the store, John gave the bag to"
corrupted_prompt = "When John and Mary went to the store, Mary gave the bag to"

clean_tokens     = model.to_tokens(clean_prompt)
corrupted_tokens = model.to_tokens(corrupted_prompt)

# Get clean and corrupted logits + cache
clean_logits,     clean_cache     = model.run_with_cache(clean_tokens)
corrupted_logits, corrupted_cache = model.run_with_cache(corrupted_tokens)

# Target tokens
mary_token = model.to_single_token(" Mary")
john_token = model.to_single_token(" John")

def logit_diff(logits):
    """Difference in logit between Mary and John at final position"""
    return (logits[0, -1, mary_token] - 
            logits[0, -1, john_token]).item()

clean_ld     = logit_diff(clean_logits)
corrupted_ld = logit_diff(corrupted_logits)

print(f"Clean prompt logit diff (Mary - John):     {clean_ld:.4f}")
print(f"Corrupted prompt logit diff (Mary - John): {corrupted_ld:.4f}")
print(f"\nA positive value means the model predicts Mary")
print(f"A negative value means the model predicts John")
print(f"\nPatching each layer to find which ones matter...")

# Patch each layer's residual stream
layer_effects = []
for layer in range(model.cfg.n_layers):
    def hook_fn(value, hook, layer=layer):
        value = clean_cache['resid_post', layer]
        return value
    
    patched_logits = model.run_with_hooks(
        corrupted_tokens,
        fwd_hooks=[(f'blocks.{layer}.hook_resid_post', hook_fn)]
    )
    patched_ld = logit_diff(patched_logits)
    layer_effects.append({
        'layer':      layer,
        'logit_diff': round(patched_ld, 4),
        'recovery':   round((patched_ld - corrupted_ld) / 
                            (clean_ld - corrupted_ld) * 100, 2)
    })
    print(f"  Layer {layer:02d}: logit_diff={patched_ld:.4f} "
          f"recovery={layer_effects[-1]['recovery']:.1f}%")

effects_df = pd.DataFrame(layer_effects)

Clean prompt logit diff (Mary - John):     3.3796
Corrupted prompt logit diff (Mary - John): -3.7383

A positive value means the model predicts Mary
A negative value means the model predicts John

Patching each layer to find which ones matter...
  Layer 00: logit_diff=3.3796 recovery=100.0%
  Layer 01: logit_diff=3.3796 recovery=100.0%
  Layer 02: logit_diff=3.3796 recovery=100.0%
  Layer 03: logit_diff=3.3796 recovery=100.0%
  Layer 04: logit_diff=3.3796 recovery=100.0%
  Layer 05: logit_diff=3.3796 recovery=100.0%
  Layer 06: logit_diff=3.3796 recovery=100.0%
  Layer 07: logit_diff=3.3796 recovery=100.0%
  Layer 08: logit_diff=3.3796 recovery=100.0%
  Layer 09: logit_diff=3.3796 recovery=100.0%
  Layer 10: logit_diff=3.3796 recovery=100.0%
  Layer 11: logit_diff=3.3796 recovery=100.0%


In [12]:
# Cell 6 — Plot Patching Results & Export
fig = go.Figure()
fig.add_trace(go.Bar(
    x=effects_df['layer'],
    y=effects_df['recovery'],
    marker=dict(
        color=effects_df['recovery'],
        colorscale='RdYlGn',
        showscale=True
    ),
    name='Recovery %'
))

fig.add_hline(y=100, line=dict(color='lime', dash='dash'),
              annotation_text='Full recovery (clean)')
fig.add_hline(y=0, line=dict(color='red', dash='dash'),
              annotation_text='No recovery (corrupted)')

fig.update_layout(
    title='Activation Patching — Which Layers Encode IOI Circuit?',
    xaxis_title='Layer',
    yaxis_title='Logit Diff Recovery (%)',
    template='plotly_dark',
    width=850, height=480
)
fig.show()

# Export
import os
output_dir = r'C:\Users\Gurveer\ds-portfolio\project-11-mechanistic-interp\outputs'
os.makedirs(output_dir, exist_ok=True)
effects_df.to_csv(f'{output_dir}\\layer_patching_results.csv', index=False)

pd.DataFrame({
    'metric': ['clean_logit_diff', 'corrupted_logit_diff'],
    'value':  [clean_ld, corrupted_ld]
}).to_csv(f'{output_dir}\\logit_diff_baseline.csv', index=False)

print(f"Top 3 most important layers:")
print(effects_df.nlargest(3, 'recovery')[
    ['layer','logit_diff','recovery']
].to_string(index=False))
print(f"\nExports saved to outputs/")

Top 3 most important layers:
 layer  logit_diff  recovery
     0      3.3796     100.0
     1      3.3796     100.0
     2      3.3796     100.0

Exports saved to outputs/
